# 📚 Scraper books.toscrape.com → MySQL
### Practica SQL con categorías, libros y autores

**Flujo del notebook:**
1. Instalar dependencias
2. Conectar a MySQL y crear tablas
3. Scrapear con hilos
4. Insertar datos
5. Consultas SQL para practicar


In [4]:
# Instalar librerías necesarias
# (solo la primera vez)
!pip install requests beautifulsoup4 mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import mysql.connector
import time
import random

## 🔌 Paso 1 — Conectar a MySQL

In [6]:
# ⚠️ Cambia estos datos por los de tu MySQL
configuracion = {
    "host":     "localhost",
    "user":     "root",
    "password": "5860464",
    "port":     3306
}

NOMBRE_BASE_DATOS = "libros_scraper"

conexion = mysql.connector.connect(
    **configuracion,
    auth_plugin='mysql_native_password'
)
cursor   = conexion.cursor()

cursor.execute(f"CREATE DATABASE IF NOT EXISTS {NOMBRE_BASE_DATOS}")
cursor.execute(f"USE {NOMBRE_BASE_DATOS}")

print(f"✅ Conectado a MySQL — base de datos: {NOMBRE_BASE_DATOS}")

✅ Conectado a MySQL — base de datos: libros_scraper


## 🗄️ Paso 2 — Crear tablas

Estructura de relaciones:
```
categorias ──< libros >── libros_autores >── autores
```
- Una categoría tiene muchos libros
- Un libro puede tener varios autores (tabla intermedia)


In [7]:
# ── Tabla: categorias ────────────────────────────────────────
cursor.execute("""
CREATE TABLE IF NOT EXISTS categorias (
    id_categoria  INT AUTO_INCREMENT PRIMARY KEY,
    nombre        VARCHAR(100) NOT NULL UNIQUE
)
""")

# ── Tabla: autores ────────────────────────────────────────────
cursor.execute("""
CREATE TABLE IF NOT EXISTS autores (
    id_autor  INT AUTO_INCREMENT PRIMARY KEY,
    nombre    VARCHAR(200) NOT NULL UNIQUE
)
""")

# ── Tabla: libros ─────────────────────────────────────────────
# → se relaciona con categorias por id_categoria
cursor.execute("""
CREATE TABLE IF NOT EXISTS libros (
    id_libro      INT AUTO_INCREMENT PRIMARY KEY,
    titulo        VARCHAR(500) NOT NULL,
    precio        DECIMAL(10,2),
    calificacion  TINYINT,
    disponible    VARCHAR(50),
    url           VARCHAR(500),
    id_categoria  INT NOT NULL,
    FOREIGN KEY (id_categoria) REFERENCES categorias(id_categoria)
)
""")

# ── Tabla intermedia: libros_autores ──────────────────────────
# → un libro puede tener varios autores y viceversa (N:M)
cursor.execute("""
CREATE TABLE IF NOT EXISTS libros_autores (
    id_libro INT,
    id_autor INT,
    PRIMARY KEY (id_libro, id_autor),
    FOREIGN KEY (id_libro) REFERENCES libros(id_libro),
    FOREIGN KEY (id_autor) REFERENCES autores(id_autor)
)
""")

conexion.commit()
print("✅ Tablas creadas: categorias, libros, autores, libros_autores")

✅ Tablas creadas: categorias, libros, autores, libros_autores


## 🕷️ Paso 3 — Scrapear con hilos

In [8]:
URL_BASE = "https://books.toscrape.com"

ESTRELLAS_A_NUMERO = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
}

# Autores de ejemplo — books.toscrape.com no tiene autores reales
# así que asignamos uno aleatorio para practicar SQL con esa relación
AUTORES_FICTICIOS = [
    "Gabriel García Márquez", "Isabel Allende", "Jorge Luis Borges",
    "Pablo Neruda", "Julio Cortázar", "Mario Vargas Llosa",
    "Laura Esquivel", "Carlos Fuentes", "Octavio Paz", "Elena Poniatowska"
]


def obtener_categorias():
    respuesta = requests.get(URL_BASE)
    soup      = BeautifulSoup(respuesta.text, "html.parser")
    
    categorias = []
    for enlace in soup.select("ul.nav-list > li > ul > li > a"):
        categorias.append({
            "nombre": enlace.text.strip(),
            "url":    URL_BASE + "/" + enlace["href"]
        })
    return categorias


def obtener_libros_de_pagina(soup):
    libros = []
    for articulo in soup.select("article.product_pod"):
        titulo       = articulo.select_one("h3 > a")["title"]
        precio_texto = articulo.select_one("p.price_color").text.strip()
        precio       = float(precio_texto.replace("£", "").replace("Â", "").strip())
        clase_stars  = articulo.select_one("p.star-rating")["class"][1]
        calificacion = ESTRELLAS_A_NUMERO.get(clase_stars, 0)
        disponible   = articulo.select_one("p.availability").text.strip()
        url_relativa = articulo.select_one("h3 > a")["href"].replace("../../../", "catalogue/")
        url_libro    = URL_BASE + "/" + url_relativa

        # Asignamos 1 o 2 autores ficticios aleatorios por libro
        autores = random.sample(AUTORES_FICTICIOS, k=random.randint(1, 2))

        libros.append({
            "titulo":       titulo,
            "precio":       precio,
            "calificacion": calificacion,
            "disponible":   disponible,
            "url":          url_libro,
            "autores":      autores
        })
    return libros


def scrapear_categoria(categoria):
    nombre    = categoria["nombre"]
    url_actual = categoria["url"]
    libros_totales = []

    try:
        while True:
            respuesta = requests.get(url_actual, timeout=10)
            soup      = BeautifulSoup(respuesta.text, "html.parser")
            libros_totales.extend(obtener_libros_de_pagina(soup))

            siguiente = soup.select_one("li.next > a")
            if siguiente:
                base       = url_actual.rsplit("/", 1)[0]
                url_actual = base + "/" + siguiente["href"]
                time.sleep(0.2)
            else:
                break

        return {"nombre": nombre, "libros": libros_totales, "error": None}

    except Exception as e:
        return {"nombre": nombre, "libros": [], "error": str(e)}


print("✅ Funciones de scraping listas")

✅ Funciones de scraping listas


In [9]:
MAX_HILOS = 5

print("🚀 Iniciando scraping con hilos...\n")
inicio   = time.time()
datos    = []

with ThreadPoolExecutor(max_workers=MAX_HILOS) as ejecutor:
    futuros = {
        ejecutor.submit(scrapear_categoria, cat): cat["nombre"]
        for cat in obtener_categorias()
    }

    for futuro in as_completed(futuros):
        resultado = futuro.result()
        if resultado["error"]:
            print(f"  ❌ Error en {resultado['nombre']}: {resultado['error']}")
        else:
            print(f"  ✅ {resultado['nombre']:<30} → {len(resultado['libros'])} libros")
            datos.append(resultado)

fin = time.time()
total_libros = sum(len(d["libros"]) for d in datos)
print(f"\n🎉 {total_libros} libros en {len(datos)} categorías — ⏱️ {fin-inicio:.1f}s")

🚀 Iniciando scraping con hilos...

  ✅ Travel                         → 11 libros
  ✅ Classics                       → 19 libros
  ✅ Philosophy                     → 11 libros
  ✅ Historical Fiction             → 26 libros
  ✅ Mystery                        → 32 libros
  ✅ Womens Fiction                 → 17 libros
  ✅ Romance                        → 35 libros
  ✅ Religion                       → 7 libros
  ✅ Childrens                      → 29 libros
  ✅ Music                          → 13 libros
  ✅ Sequential Art                 → 75 libros
  ✅ Sports and Games               → 5 libros
  ✅ Science Fiction                → 16 libros
  ✅ Fiction                        → 65 libros
  ✅ New Adult                      → 6 libros
  ✅ Fantasy                        → 48 libros
  ✅ Science                        → 14 libros
  ✅ Add a comment                  → 67 libros
  ✅ Nonfiction                     → 110 libros
  ✅ Young Adult                    → 54 libros
  ✅ Paranormal             

## 💾 Paso 4 — Insertar en MySQL

In [10]:
def obtener_o_crear_id(tabla, columna_id, columna_nombre, valor):
    """Inserta un registro si no existe y devuelve su id"""
    cursor.execute(
        f"SELECT {columna_id} FROM {tabla} WHERE {columna_nombre} = %s",
        (valor,)
    )
    fila = cursor.fetchone()
    if fila:
        return fila[0]
    
    cursor.execute(
        f"INSERT INTO {tabla} ({columna_nombre}) VALUES (%s)",
        (valor,)
    )
    conexion.commit()
    return cursor.lastrowid


print("💾 Insertando datos...\n")

for categoria_data in datos:
    # Insertar categoría y obtener su id
    id_categoria = obtener_o_crear_id(
        "categorias", "id_categoria", "nombre", categoria_data["nombre"]
    )

    for libro in categoria_data["libros"]:
        # Insertar libro
        cursor.execute("""
            INSERT INTO libros (titulo, precio, calificacion, disponible, url, id_categoria)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            libro["titulo"],
            libro["precio"],
            libro["calificacion"],
            libro["disponible"],
            libro["url"],
            id_categoria
        ))
        id_libro = cursor.lastrowid

        # Insertar autores y la relación libros_autores
        for nombre_autor in libro["autores"]:
            id_autor = obtener_o_crear_id(
                "autores", "id_autor", "nombre", nombre_autor
            )
            cursor.execute("""
                INSERT IGNORE INTO libros_autores (id_libro, id_autor)
                VALUES (%s, %s)
            """, (id_libro, id_autor))

    conexion.commit()
    print(f"  ✅ {categoria_data['nombre']}")

print("\n🎉 ¡Todos los datos insertados!")

💾 Insertando datos...

  ✅ Travel
  ✅ Classics
  ✅ Philosophy
  ✅ Historical Fiction
  ✅ Mystery
  ✅ Womens Fiction
  ✅ Romance
  ✅ Religion
  ✅ Childrens
  ✅ Music
  ✅ Sequential Art
  ✅ Sports and Games
  ✅ Science Fiction
  ✅ Fiction
  ✅ New Adult
  ✅ Fantasy
  ✅ Science
  ✅ Add a comment
  ✅ Nonfiction
  ✅ Young Adult
  ✅ Paranormal
  ✅ Poetry
  ✅ Art
  ✅ Psychology
  ✅ Autobiography
  ✅ Parenting
  ✅ Adult Fiction
  ✅ Humor
  ✅ Horror
  ✅ History
  ✅ Christian Fiction
  ✅ Business
  ✅ Biography
  ✅ Default
  ✅ Food and Drink
  ✅ Thriller
  ✅ Contemporary
  ✅ Spirituality
  ✅ Historical
  ✅ Academic
  ✅ Self Help
  ✅ Christian
  ✅ Suspense
  ✅ Novels
  ✅ Health
  ✅ Short Stories
  ✅ Politics
  ✅ Cultural
  ✅ Crime
  ✅ Erotica

🎉 ¡Todos los datos insertados!


## 🔍 Paso 5 — Consultas SQL para practicar

Desde aquí en adelante es puro SQL. ¡Modifica las consultas a tu gusto!

In [11]:
# 📂 ¿Cuántos libros tiene cada categoría?
cursor.execute("""
    SELECT c.nombre, COUNT(l.id_libro) AS total_libros
    FROM categorias c
    JOIN libros l ON l.id_categoria = c.id_categoria
    GROUP BY c.nombre
    ORDER BY total_libros DESC
""")
print(f"{'Categoría':<30} {'Libros':>8}")
print("─" * 40)
for fila in cursor.fetchall():
    print(f"{fila[0]:<30} {fila[1]:>8}")

Categoría                        Libros
────────────────────────────────────────
Default                             304
Nonfiction                          220
Sequential Art                      150
Add a comment                       134
Fiction                             130
Young Adult                         108
Fantasy                              96
Romance                              70
Mystery                              64
Food and Drink                       60
Childrens                            58
Historical Fiction                   52
Classics                             38
Poetry                               38
History                              36
Horror                               34
Womens Fiction                       34
Science Fiction                      32
Science                              28
Music                                26
Business                             24
Philosophy                           22
Thriller                             22

In [12]:
# ⭐ Libros con 5 estrellas y su categoría
cursor.execute("""
    SELECT l.titulo, l.precio, c.nombre AS categoria
    FROM libros l
    JOIN categorias c ON c.id_categoria = l.id_categoria
    WHERE l.calificacion = 5
    LIMIT 10
""")
print(f"{'Título':<45} {'Precio':>8}  {'Categoría'}")
print("─" * 75)
for fila in cursor.fetchall():
    print(f"{fila[0][:44]:<45} £{fila[1]:>7}  {fila[2]}")

Título                                          Precio  Categoría
───────────────────────────────────────────────────────────────────────────
1,000 Places to See Before You Die            £  26.08  Travel
Sophie's World                                £  15.94  Philosophy
At The Existentialist CafÃ©: Freedom, Being,  £  29.93  Philosophy
A Flight of Arrows (The Pathfinders #2)       £  55.53  Historical Fiction
Mrs. Houdini                                  £  30.25  Historical Fiction
The Passion of Dolssa                         £  28.32  Historical Fiction
Voyager (Outlander #3)                        £  21.07  Historical Fiction
The Red Tent                                  £  35.66  Historical Fiction
Between Shades of Gray                        £  20.79  Historical Fiction
While You Were Mine                           £  41.32  Historical Fiction


In [13]:
# 💰 Precio promedio por categoría
cursor.execute("""
    SELECT c.nombre, ROUND(AVG(l.precio), 2) AS precio_promedio
    FROM categorias c
    JOIN libros l ON l.id_categoria = c.id_categoria
    GROUP BY c.nombre
    ORDER BY precio_promedio DESC
    LIMIT 10
""")
print(f"{'Categoría':<30} {'Precio promedio':>15}")
print("─" * 47)
for fila in cursor.fetchall():
    print(f"{fila[0]:<30} £{fila[1]:>14}")

Categoría                      Precio promedio
───────────────────────────────────────────────
Suspense                       £         58.33
Novels                         £         54.81
Politics                       £         53.61
Health                         £         51.45
New Adult                      £         46.38
Christian                      £         42.50
Sports and Games               £         41.17
Self Help                      £         40.62
Travel                         £         39.79
Fantasy                        £         39.59


In [14]:
# 🧑‍💼 ¿Cuántos libros tiene cada autor? (relación N:M)
cursor.execute("""
    SELECT a.nombre, COUNT(la.id_libro) AS total_libros
    FROM autores a
    JOIN libros_autores la ON la.id_autor = a.id_autor
    GROUP BY a.nombre
    ORDER BY total_libros DESC
""")
print(f"{'Autor':<35} {'Libros':>8}")
print("─" * 45)
for fila in cursor.fetchall():
    print(f"{fila[0]:<35} {fila[1]:>8}")

Autor                                 Libros
─────────────────────────────────────────────
Octavio Paz                              329
Jorge Luis Borges                        319
Mario Vargas Llosa                       308
Pablo Neruda                             308
Isabel Allende                           302
Carlos Fuentes                           301
Laura Esquivel                           293
Elena Poniatowska                        291
Gabriel García Márquez                   287
Julio Cortázar                           286


In [15]:
# 📖 Ver los autores de un libro específico
cursor.execute("""
    SELECT l.titulo, GROUP_CONCAT(a.nombre SEPARATOR ', ') AS autores
    FROM libros l
    JOIN libros_autores la ON la.id_libro = l.id_libro
    JOIN autores a ON a.id_autor = la.id_autor
    GROUP BY l.id_libro
    LIMIT 10
""")
print(f"{'Título':<45}  Autores")
print("─" * 80)
for fila in cursor.fetchall():
    print(f"{fila[0][:44]:<45}  {fila[1]}")

Título                                         Autores
────────────────────────────────────────────────────────────────────────────────
It's Only the Himalayas                        Elena Poniatowska
Full Moon over Noahâs Ark: An Odyssey to M   Laura Esquivel, Julio Cortázar
See America: A Celebration of Our National P   Isabel Allende
Vagabonding: An Uncommon Guide to the Art of   Julio Cortázar
Under the Tuscan Sun                           Gabriel García Márquez
A Summer In Europe                             Elena Poniatowska, Gabriel García Márquez
The Great Railway Bazaar                       Isabel Allende, Jorge Luis Borges
A Year in Provence (Provence #1)               Isabel Allende, Jorge Luis Borges
The Road to Little Dribbling: Adventures of    Gabriel García Márquez
Neither Here nor There: Travels in Europe      Gabriel García Márquez, Mario Vargas Llosa


In [16]:
# Cerrar conexión al terminar
cursor.close()
conexion.close()
print("🔒 Conexión cerrada")

🔒 Conexión cerrada
